# Qwen3 Baseline Evaluation Server

Colab GPU에서 **vLLM**으로 Qwen3 모델을 서빙하고, 로컬 평가 프레임워크(`logical-puzzles`)에서 HTTP로 호출합니다.
vLLM의 Continuous Batching으로 다수의 동시 요청을 효율적으로 처리합니다.

## 빠른 시작
1. **런타임 → A100 GPU 설정**
2. **Cell 1 ~ 4 순서대로 실행** (Cell 2에서 ngrok 토큰 입력)
3. Cell 4 출력의 URL을 쉘 스크립트 `REMOTE_URL`에 설정
4. 로컬에서 평가 실행:
   ```bash
   bash scripts/eval_remote.sh
   ```

## 지원 모델
| 모델 | 권장 GPU |
|---|---|
| `Qwen/Qwen3-0.6B` | T4 / A100 |
| `Qwen/Qwen3-1.7B` | T4 / A100 |

## API (OpenAI 호환)
- **엔드포인트**: `POST /v1/chat/completions`
- **요청**: OpenAI Chat Completion 형식
- **응답**: OpenAI 형식 + `reasoning_content` (thinking 내용)

In [ ]:
# [1] vLLM + ngrok 설치
!pip uninstall -yq google-adk opentelemetry-exporter-gcp-logging 2>/dev/null
!pip install -q vllm pyngrok jedi

In [ ]:
# [2] ngrok 인증
# 토큰 발급: https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = "3BF1C6UMNiyXyNBY7XfAKRFCOpD_7BiXpaX7pYxspr84E22Zy" # 준용
# NGROK_AUTH_TOKEN = "your-authtoken"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok authentication complete")

In [ ]:
# [3] vLLM 서버 시작 (백그라운드)
import subprocess, sys, time

MODEL = "Qwen/Qwen3-0.6B"
# MODEL = "Qwen/Qwen3-1.7B"

proc = subprocess.Popen(
    [
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--host", "0.0.0.0",
        "--port", "8000",
        "--dtype", "bfloat16",
        "--max-model-len", "32768",
        "--trust-remote-code",
        "--reasoning-parser", "deepseek_r1",
    ],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
)

import requests as _req
print(f"Starting vLLM server... (Model: {MODEL})")
for i in range(300):
    try:
        r = _req.get("http://localhost:8000/health")
        if r.status_code == 200:
            print(f"vLLM server is ready ({i+1} sec)")
            break
    except Exception:
        pass
    time.sleep(1)
    if (i + 1) % 15 == 0:
        print(f"  Waiting... ({i+1} sec)")
else:
    print("Server failed to start! Log:")
    with open("/tmp/vllm.log") as f:
        print(f.read()[-3000:])

In [ ]:
# [4] ngrok 연결 + URL 출력
public_url = ngrok.connect(8000)

print(f"\n{'★' * 50}")
print(f"  Remote URL: {public_url.public_url}")
print(f"  엔드포인트: {public_url.public_url}/v1/chat/completions")
print(f"{'★' * 50}")
print(f"\n로컬 쉘 스크립트의 REMOTE_URL을 위 주소로 변경하세요.")
print(f'예: REMOTE_URL="{public_url.public_url}"')

In [ ]:
# [5] vLLM 서버 로그 실시간 확인
!tail -f /tmp/vllm.log